# 🧠 Notebook 3 — QLoRA Fine-tuning

**Author:** Maulik Mathur | **HuggingFace:** [maulik78](https://huggingface.co/maulik78)

---

## Why This Notebook Exists

Notebook 2 showed that XGBoost achieves **$68.23 MAE** using word-count features.
The model treats words independently — it sees 'Sony' and 'noise' and 'cancelling'
as three separate signals. It doesn't understand that together they mean
'premium consumer audio product'.

Can a language model do better by understanding context?

**Answer: Yes. Fine-tuned Llama 3.2-3B achieves $58.74 MAE — 13.9% better.**

## The Fine-tuning Approach: QLoRA

Full fine-tuning of Llama 3.2-3B would require:
- ~24GB VRAM (we have 15GB on free T4)
- Update all 3 billion parameters
- Store gradients for 3B params (another 12GB)

QLoRA solves this in two ways:

```
Q = Quantization: compress model weights
    float32 (4 bytes/param) → 4-bit NF4 (0.5 bytes/param)
    Memory: 6.4GB → 2.2GB  (3x reduction)

LoRA = Low-Rank Adaptation: train tiny adapters
    Instead of updating W (3072×3072 = 9.4M params)
    We train A (3072×32) and B (32×3072) = 196k params
    Memory: 3B trainable params → 18M trainable params

Result: fits on free T4 GPU in under 2 hours
```

## Runtime
- **Platform:** Google Colab (T4 GPU — free)
- **Time:** ~2 hours
- **Cost:** Free
- **Output:** Your model at huggingface.co/maulik78/pricer-[timestamp]

## Step 1 — Install Dependencies

We pin specific versions because ML libraries change rapidly and
newer versions sometimes break compatibility with each other.

In [ ]:
# Pin versions for reproducibility
# bitsandbytes: enables 4-bit quantization
# trl: provides SFTTrainer (Supervised Fine-Tuning Trainer)
# peft: provides LoRA implementation
!pip install -q --upgrade bitsandbytes==0.48.2 trl==0.25.1 peft transformers accelerate

import os, sys, torch
from datetime import datetime
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    set_seed
)
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig
import wandb

# Check what GPU we have
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

# bfloat16 is more stable than float16 but only available on A100+
# T4 (compute capability 7.5) must use float16
capability = torch.cuda.get_device_capability()
USE_BF16   = capability[0] >= 8
print(f'Compute capability: {capability} → Using {"bfloat16" if USE_BF16 else "float16"}')

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'), add_to_git_credential=True)

# Optional: Weights & Biases for training visualization
# Add WANDB_API_KEY to Colab Secrets if you have a W&B account
try:
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
    wandb.login()
    LOG_WANDB = True
    print('✅ W&B logging enabled')
except:
    LOG_WANDB = False
    print('W&B not configured — skipping experiment tracking')

print('✅ Authentication complete')

---
## Part 1 — Training Configuration

Every hyperparameter here has a reason. Read the comments carefully —
these are the decisions you'll explain in interviews.

In [ ]:
# ── Model and Data ────────────────────────────────────────────
BASE_MODEL   = 'meta-llama/Llama-3.2-3B'
HF_USERNAME  = 'maulik78'
DATASET_NAME = f'{HF_USERNAME}/items_prompts_lite'  # 20k training examples

# ── Run naming ────────────────────────────────────────────────
# Timestamp in name = every run is uniquely identifiable
# Useful when comparing multiple training runs
RUN_NAME         = f"{datetime.now():%Y-%m-%d_%H.%M.%S}-lite"
MODEL_SAVE_NAME  = f"{HF_USERNAME}/pricer-{RUN_NAME}"

print(f'Model will be saved as: {MODEL_SAVE_NAME}')
print(f'Dataset: {DATASET_NAME}')

In [ ]:
# ── QLoRA Hyperparameters ──────────────────────────────────────

# Quantization: 4-bit NF4
# WHY NF4 over standard int4:
#   Neural network weights follow a normal (bell curve) distribution.
#   Standard int4 divides the range into 16 equal buckets — wastes
#   buckets on rare extreme values.
#   NF4 places buckets by normal distribution quantiles — more
#   buckets near zero where weights cluster. Better precision.
LOAD_IN_4BIT = True

# LoRA rank (r)
# WHY r=32:
#   r controls how many parameters LoRA trains.
#   r=32 → 18M trainable params (0.6% of 3B total)
#   r=256 → 389M trainable params (13% of 3B total)
#   r=32 is sufficient for task-specific adaptation.
#   Higher r = more capacity but also more overfitting risk
#   on our small 20k dataset.
LORA_RANK    = 32

# LoRA alpha: scaling factor for the adapter
# WHY alpha = 2 * rank:
#   The effective scale of LoRA updates is alpha/rank.
#   Setting alpha = 2*rank means scale = 2.0
#   This is a well-established default from the original LoRA paper.
LORA_ALPHA   = LORA_RANK * 2  # = 64

# LoRA dropout: regularization to prevent overfitting
# WHY 0.1:
#   10% of adapter activations are randomly zeroed during training.
#   This prevents the adapter from memorizing specific products.
LORA_DROPOUT = 0.1

# Target modules: which layers get LoRA adapters
# WHY only attention layers:
#   In a Transformer, attention layers learn RELATIONSHIPS between tokens.
#   This is where pricing knowledge lives — understanding that
#   'Sony + wireless + noise-cancelling' = high price.
#   MLP layers learn per-token transformations — less important for
#   our domain adaptation task.
TARGET_MODULES = ['q_proj', 'v_proj', 'k_proj', 'o_proj']

print('QLoRA Configuration:')
print(f'  Quantization: 4-bit NF4')
print(f'  LoRA rank (r): {LORA_RANK}')
print(f'  LoRA alpha:    {LORA_ALPHA} (scale = {LORA_ALPHA/LORA_RANK:.1f})')
print(f'  Dropout:       {LORA_DROPOUT}')
print(f'  Target layers: {TARGET_MODULES}')

In [ ]:
# ── Training Hyperparameters ───────────────────────────────────

# Batch size: how many examples per gradient update
# WHY batch_size=4 (not 32 like normal):
#   The full batch (4 × 32 = 128 tokens each) needs to fit in GPU RAM
#   alongside the model (2.2GB), optimizer states, and activations.
#   We compensate with gradient accumulation (see below).
BATCH_SIZE = 4

# Gradient accumulation: accumulate gradients over N batches
# WHY accumulation=8:
#   Effective batch size = 4 × 8 = 32
#   This simulates training with a batch of 32 while only loading
#   4 examples at a time. Same result, less memory.
GRADIENT_ACCUMULATION = 8

# Epochs: how many times to pass through the training data
# WHY 1 epoch:
#   With only 20k training examples, more than 1 epoch risks
#   overfitting. The model starts memorizing specific products
#   rather than learning general pricing patterns.
EPOCHS = 1

# Learning rate: step size for weight updates
# WHY 1e-4:
#   Standard for LoRA fine-tuning. Large enough to learn quickly,
#   small enough to not corrupt pretrained knowledge.
LEARNING_RATE = 1e-4

# Warmup: gradually increase LR at the start of training
# WHY 1% warmup:
#   At the very start, gradients are unstable. A sudden large LR
#   can damage the pretrained weights. We ramp up over 1% of steps.
WARMUP_RATIO = 0.01

# Cosine LR scheduler: decay LR following a cosine curve
# WHY cosine over constant:
#   Starts at 1e-4, gradually decreases toward 0 by end of training.
#   Models converge better with a decaying learning rate.
LR_SCHEDULER = 'cosine'

# Gradient clipping: cap gradient magnitude
# WHY 0.3:
#   Prevents 'exploding gradients' — occasional very large gradient
#   steps that can damage learned weights. 0.3 is aggressive clipping
#   suitable for fine-tuning (vs 1.0 for training from scratch).
MAX_GRAD_NORM = 0.3

# Optimizer: AdamW variant designed for limited GPU memory
# WHY paged_adamw_32bit:
#   Regular Adam stores 2 momentum vectors per parameter in GPU RAM.
#   Paged Adam offloads these to CPU RAM when GPU is full.
#   Saves ~2GB GPU memory vs regular Adam.
OPTIMIZER = 'paged_adamw_32bit'

MAX_SEQ_LEN  = 128  # max tokens per training example
SAVE_STEPS   = 100  # save checkpoint every 100 steps
LOG_STEPS    = 5    # log metrics every 5 steps
VAL_SIZE     = 500  # use 500 validation items for monitoring

print('Training Configuration:')
print(f'  Batch size:    {BATCH_SIZE} × {GRADIENT_ACCUMULATION} accumulation = {BATCH_SIZE*GRADIENT_ACCUMULATION} effective')
print(f'  Epochs:        {EPOCHS}')
print(f'  Learning rate: {LEARNING_RATE}')
print(f'  LR schedule:   {LR_SCHEDULER} with {WARMUP_RATIO*100:.0f}% warmup')
print(f'  Max grad norm: {MAX_GRAD_NORM}')

---
## Part 2 — Understanding the Training Data

Before training, let's understand exactly what the model will see.

In [ ]:
dataset = load_dataset(DATASET_NAME)

train = dataset['train']
val   = dataset['val'].select(range(VAL_SIZE))
test  = dataset['test']

print(f'Dataset: {DATASET_NAME}')
print(f'  Train: {len(train):,} examples')
print(f'  Val:   {len(val):,} examples (using {VAL_SIZE} for monitoring)')
print(f'  Test:  {len(test):,} examples (held out for final evaluation)')
print(f'  Fields: {train.column_names}')

In [ ]:
# Understand the prompt format
# CRITICAL: The model learns to complete these exact prompt patterns
# At inference, we give the prompt WITHOUT the completion
# The model must generate the price

print('TRAINING EXAMPLE (model sees both prompt + completion):')
print('='*65)
print(train[0]['prompt'])
print()
print(f'COMPLETION (target the model learns to generate):')
print(f'  "{train[0]["completion"]}"')
print()
print('INFERENCE (model sees only the prompt, generates completion):')
print(train[0]['prompt'].split('Price is $')[0] + 'Price is $[MODEL GENERATES THIS]')

In [ ]:
# Check prompt length distribution
# MAX_SEQ_LEN=128 should cover most examples without truncation
import matplotlib.pyplot as plt

lengths = [len(ex['prompt'].split()) for ex in train.select(range(5000))]

plt.figure(figsize=(12, 4))
plt.hist(lengths, bins=50, color='steelblue', rwidth=0.8)
plt.axvline(x=MAX_SEQ_LEN//4, color='red', linestyle='--', label=f'MAX_SEQ_LEN≈{MAX_SEQ_LEN} tokens')
plt.title(f'Prompt Length Distribution (words)\nMax seq len={MAX_SEQ_LEN} tokens')
plt.xlabel('Words in prompt')
plt.ylabel('Count')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Prompt length stats (words):')
print(f'  Mean:   {sum(lengths)/len(lengths):.1f}')
print(f'  Median: {sorted(lengths)[len(lengths)//2]}')
print(f'  Max:    {max(lengths)}')

---
## Part 3 — Load the Model

Two-step loading:
1. Load frozen base model (4-bit quantized)
2. Attach trainable LoRA adapters

In [ ]:
# Free any previous model from GPU
import gc
try:
    del model
    gc.collect()
    torch.cuda.empty_cache()
except:
    pass

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# 4-bit quantization configuration
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,     # quantize the quantization constants
    bnb_4bit_compute_dtype=torch.bfloat16 if USE_BF16 else torch.float16,
    bnb_4bit_quant_type='nf4'           # NormalFloat4 format
)

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token    = tokenizer.eos_token  # Llama has no pad token
tokenizer.padding_side = 'right'              # pad on right for SFT

print('Loading Llama 3.2-3B in 4-bit...')
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map='auto'  # HuggingFace places layers across available devices
)
model.generation_config.pad_token_id = tokenizer.eos_token_id

memory_mb = model.get_memory_footprint() / 1e6
print(f'✅ Base model loaded')
print(f'   Memory footprint: {memory_mb:.0f} MB  (vs ~6,400 MB in float32)')

In [ ]:
# Define LoRA configuration
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',          # don't add LoRA to bias terms
    task_type='CAUSAL_LM',
    target_modules=TARGET_MODULES
)

# Attach LoRA to see parameter counts
temp_model  = get_peft_model(model, lora_config)
trainable   = sum(p.numel() for p in temp_model.parameters() if p.requires_grad)
total       = sum(p.numel() for p in temp_model.parameters())
del temp_model
gc.collect()

print(f'LoRA Parameter Analysis:')
print(f'  Total parameters:     {total:,}  ({total/1e9:.2f}B)')
print(f'  Trainable (LoRA):     {trainable:,}  ({100*trainable/total:.2f}%)')
print(f'  Frozen (base model):  {total-trainable:,}  ({100*(total-trainable)/total:.2f}%)')
print(f'\n  Only {trainable/1e6:.1f}M parameters will be updated during training')
print(f'  The other {(total-trainable)/1e9:.2f}B parameters stay frozen')

---
## Part 4 — Train

The `SFTTrainer` (Supervised Fine-Tuning Trainer) from TRL handles:
- Applying LoRA adapters to the model
- The training loop (forward pass, loss, backward, optimizer step)
- Checkpointing and pushing to HuggingFace
- Logging to W&B

During training you'll see loss decreasing — this means the model
is learning to predict prices better.

In [ ]:
# Training configuration
train_config = SFTConfig(
    output_dir=f'pricer-{RUN_NAME}',
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    optim=OPTIMIZER,
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    logging_steps=LOG_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.001,
    fp16=not USE_BF16,
    bf16=USE_BF16,
    max_grad_norm=MAX_GRAD_NORM,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type=LR_SCHEDULER,
    report_to='wandb' if LOG_WANDB else 'none',
    run_name=RUN_NAME,
    max_length=MAX_SEQ_LEN,
    save_strategy='steps',
    hub_strategy='every_save',    # push to HuggingFace at each checkpoint
    push_to_hub=True,             # enables automatic HF pushing
    hub_model_id=MODEL_SAVE_NAME,
    hub_private_repo=False,
    eval_strategy='steps',
    eval_steps=SAVE_STEPS,
    dataset_text_field='prompt',  # column name in dataset to use as input
)
print('✅ Training config created')

In [ ]:
# Create trainer
if LOG_WANDB:
    wandb.init(project='amazon-price-predictor', name=RUN_NAME)

trainer = SFTTrainer(
    model=model,
    train_dataset=train,
    eval_dataset=val,
    peft_config=lora_config,
    args=train_config,
)

print('✅ Trainer created')
print(f'\nTraining summary:')
print(f'  Steps per epoch: {len(train) // (BATCH_SIZE * GRADIENT_ACCUMULATION)}')
print(f'  Total steps:     {EPOCHS * len(train) // (BATCH_SIZE * GRADIENT_ACCUMULATION)}')
print(f'  Checkpoints at:  every {SAVE_STEPS} steps')
print(f'  Pushing to:      {MODEL_SAVE_NAME}')

In [ ]:
# START TRAINING
# Watch the val/loss column — it should decrease over time
# If val loss increases while train loss decreases = overfitting

print('🚀 Starting training...')
print('   Expected time: ~2 hours on T4 GPU')
print('   Checkpoints push to HuggingFace every 100 steps')
print('   (Your model appears on HuggingFace mid-training!)')
print()

trainer.train()

# Push final model
trainer.model.push_to_hub(MODEL_SAVE_NAME, private=False)

if LOG_WANDB:
    wandb.finish()

print(f'\n✅ Training complete!')
print(f'   Model saved to: huggingface.co/{MODEL_SAVE_NAME}')
print(f'\n   SAVE THIS MODEL NAME:')
print(f'   FINETUNED_MODEL = "{MODEL_SAVE_NAME}"')

---
## Part 5 — Evaluate the Fine-tuned Model

Now we test whether the fine-tuned model actually learned to predict prices.

In [ ]:
# Load fine-tuned model for inference
from peft import PeftModel
import re

fine_tuned = PeftModel.from_pretrained(model, MODEL_SAVE_NAME)
fine_tuned.eval()
print(f'✅ Fine-tuned model loaded')

In [ ]:
def predict_price(prompt: str) -> float:
    """
    Generate a price prediction for a given prompt.
    
    Steps:
    1. Tokenize the prompt (text → token IDs)
    2. Generate continuation (model predicts next 8 tokens)
    3. Slice off prompt tokens (keep only generated part)
    4. Decode (token IDs → text)
    5. Parse the price number from generated text
    """
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    
    with torch.no_grad():
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16 if USE_BF16 else torch.float16):
            output_ids = fine_tuned.generate(
                **inputs,
                max_new_tokens=8,         # prices are short: "278.00" = ~4 tokens
                do_sample=False,          # deterministic: always pick highest prob token
                pad_token_id=tokenizer.eos_token_id
            )
    
    # Slice off the prompt — keep only generated tokens
    prompt_len  = inputs['input_ids'].shape[1]
    generated   = output_ids[0, prompt_len:]
    result_text = tokenizer.decode(generated, skip_special_tokens=True)
    
    # Parse number from generated text
    cleaned = result_text.replace('$','').replace(',','').strip()
    match   = re.search(r'\d+\.?\d*', cleaned)
    return float(match.group()) if match else 0.0

print('✅ Prediction function ready')

In [ ]:
# Quick test on 5 items
print('Quick test on 5 test items:')
print('='*60)

for i in range(5):
    item          = test[i]
    actual        = float(item['completion'])
    predicted     = predict_price(item['prompt'])
    error         = abs(predicted - actual)
    status        = '✅' if error < 50 else '⚠️'
    
    print(f'{status} Item {i+1}:')
    print(f'   Actual:    ${actual:.2f}')
    print(f'   Predicted: ${predicted:.2f}')
    print(f'   Error:     ${error:.2f}')
    print()

In [ ]:
# Full evaluation on 200 test items
import random
random.seed(42)

indices = random.sample(range(len(test)), 200)
errors  = []

for i in indices:
    item      = test[i]
    actual    = float(item['completion'])
    predicted = predict_price(item['prompt'])
    errors.append(abs(predicted - actual))

mae = sum(errors) / len(errors)

print('\n' + '='*60)
print('FINE-TUNING RESULTS')
print('='*60)
print(f'Model:    {MODEL_SAVE_NAME}')
print(f'Training: 20k items, 1 epoch, T4 GPU')
print(f'Test set: 200 items')
print()
print(f'{"Model":<40} {"MAE ($)":>8}')
print('-'*50)
print(f'{"XGBoost (Notebook 2 baseline)":<40} ${68.23:>7.2f}')
print(f'{"Fine-tuned Llama 3.2-3B (this model)":<40} ${mae:>7.2f}')
improvement = 68.23 - mae
print('-'*50)
print(f'Improvement over XGBoost: ${improvement:.2f} ({100*improvement/68.23:.1f}%)')
print('='*60)
print(f'\n✅ Your model: huggingface.co/{MODEL_SAVE_NAME}')
print('\n➡️  Next: Notebook 4 — Deployment and Agents')